In [ ]:


# ================== 0) Mount & Imports ==================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random, pickle, zipfile
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

GN_GROUPS = 32

def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

# ================== 1) ResNet18 Backbone + Single Head ==================
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(
        in_planes, out_planes,
        kernel_size=3, stride=stride,
        padding=1, bias=False
    )

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes, planes * self.expansion,
                    kernel_size=1, stride=stride, bias=False
                ),
                make_gn(planes * self.expansion)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        block = BasicBlock
        num_blocks = [2, 2, 2, 2]
        self.expansion = block.expansion
        self.nf = nf
        self.in_planes = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = self._make_layer(block, nf,     num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, nf * 2, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, nf * 4, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, nf * 8, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        in_planes = self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self):
        return self.nf * 8 * self.expansion  # 512

class SingleHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone, num_classes=20):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.head(feat)
        return logits

# ================== 2) Paths ==================
BASE = "/content/drive/MyDrive/ML_Project/project_files/QDA_Tiny_Imagenet"
os.makedirs(BASE, exist_ok=True)

ZIP_PATH = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

MODEL_PATH = os.path.join(BASE, "best_tinyimg_qda_20class.pth")

SAVE_TOPK_PATH      = os.path.join(BASE, "Fisher_QDA_tinyimg_20class_topk.pkl")
SAVE_NEIGHBORS_PATH = os.path.join(BASE, "Fisher_QDA_neighbors_tinyimg_20class.pkl")

TOP_K = 900000
NUM_FISHER_SAMPLES = 2000
BATCH_SIZE = 32
NUM_WORKERS = 0

# ================== 3) Tiny ImageNet processed .npy ==================
def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")

    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root

    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found at:\n{zip_path}")

    os.makedirs(data_root, exist_ok=True)
    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)

    processed_dir = os.path.join(data_root, "processed")
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    print(f"[INFO] Extracted. processed/ ready at: {processed_dir}")
    return data_root

DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):
    """
    Expects:
      processed/x_train_01.npy ... x_train_20.npy
      processed/y_train_01.npy ... y_train_20.npy
      processed/x_val_01.npy   ... x_val_20.npy
      processed/y_val_01.npy   ... y_val_20.npy
    """
    def __init__(self, root: str, train: bool = True, transform=None):
        self.root = root
        self.train = train
        self.transform = transform

        split = "train" if self.train else "val"
        xs, ys = [], []

        for num in range(20):
            xs.append(np.load(os.path.join(root, f"processed/x_{split}_{num+1:02d}.npy")))
            ys.append(np.load(os.path.join(root, f"processed/y_{split}_{num+1:02d}.npy")))

        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])
        img = Image.fromarray(np.uint8(255 * img))
        if self.transform is not None:
            img = self.transform(img)
        return img, target

tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])

train_full = TinyImagenet(DATA_ROOT, train=True, transform=tf)

keep_classes = list(range(20))
idx_20 = [i for i, t in enumerate(train_full.targets) if int(t) in keep_classes]
train_20 = Subset(train_full, idx_20)

def get_random_subset(dataset, n_samples=2000, seed=42):
    rng = random.Random(seed)
    n = len(dataset)
    k = min(n_samples, n)
    idxs = rng.sample(range(n), k)
    return Subset(dataset, idxs)

subset = get_random_subset(train_20, n_samples=NUM_FISHER_SAMPLES, seed=SEED)
subset_loader = DataLoader(
    subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

# ================== 5) Tools (ravel/unravel) ==================
def unravel_index(index, shape):
    dims = []
    for s in reversed(shape):
        dims.append(index % s)
        index //= s
    return tuple(reversed(dims))

def ravel_index(multi_idx, shape):
    flat = 0
    for idx, dim in zip(multi_idx, shape):
        flat = flat * dim + idx
    return flat

# ================== 6) Fisher computation ==================
logsoft = nn.LogSoftmax(dim=1)

def compute_fisher_per_sample_ewc_weighted(model, dataloader, device):
    """
    EWC-weighted diagonal Fisher (per-sample):
      fisher += p(y|x) * (grad_theta log p(y|x))^2
    averaged over samples

    backbone only (head excluded)
    """
    model.eval()

    fisher = {
        n: torch.zeros_like(p, device=device)
        for n, p in model.named_parameters()
        if p.requires_grad and (not n.startswith("head"))
    }

    total_samples = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        for ex, lab in zip(inputs, targets):
            ex = ex.unsqueeze(0)
            lab = lab.unsqueeze(0)

            model.zero_grad(set_to_none=True)

            logits = model(ex)     # [1,20]
            log_probs = logsoft(logits)
            logp_y = log_probs.gather(1, lab.view(1, 1)).squeeze(1)
            p_y = torch.exp(logp_y.detach())

            loss = (-logp_y).mean()
            loss.backward()

            w = float(p_y.mean().item())

            for name, p in model.named_parameters():
                if name.startswith("head"):
                    continue
                if p.grad is not None:
                    fisher[name] += w * (p.grad.detach() ** 2)

            total_samples += 1

    for name in fisher:
        fisher[name] /= max(1, total_samples)

    return fisher

# ================== 7) Top-K & Neighbors ==================
def get_topk_fisher_weights(fisher_dict, model_state_dict, k):
    if k <= 0:
        return []

    all_entries = []
    for name, f in fisher_dict.items():
        flat_f = f.flatten()
        flat_w = model_state_dict[name].flatten()

        for i in range(flat_f.numel()):
            all_entries.append({
                "name": name,
                "index": i,
                "value": float(flat_w[i].item()),
                "fisher": float(flat_f[i].item())
            })

    all_entries.sort(key=lambda x: x["fisher"], reverse=True)
    return all_entries[:k]

def extract_conv_neighbors(topk_entries, model, fisher_dict):
    neighbors = []
    if not topk_entries:
        return neighbors

    param_shapes = {name: p.shape for name, p in model.named_parameters()}
    fisher_flat = {name: tens.flatten() for name, tens in fisher_dict.items()}

    neighbor_offsets = [
        (0,0,-1,-1),(0,0,-1,1),(0,0,1,-1),(0,0,1,1),
        (0,0,-1,0),(0,0,1,0),(0,0,0,-1),(0,0,0,1),
    ]

    topk_set = set((e["name"], e["index"]) for e in topk_entries)
    added = set()

    for e in topk_entries:
        name, flat_idx = e["name"], e["index"]
        shape = param_shapes[name]

        if len(shape) != 4:
            continue

        oc, ic, kh, kw = unravel_index(flat_idx, shape)

        for do, di, dh, dw in neighbor_offsets:
            no, ni, nh, nw = oc + do, ic + di, kh + dh, kw + dw
            if 0 <= no < shape[0] and 0 <= ni < shape[1] and 0 <= nh < shape[2] and 0 <= nw < shape[3]:
                n_flat = ravel_index((no, ni, nh, nw), shape)
                key = (name, n_flat)

                if key in topk_set or key in added:
                    continue

                neighbors.append({
                    "name": name,
                    "index": n_flat,
                    "position": (no, ni, nh, nw),
                    "fisher": float(fisher_flat[name][n_flat].item())
                })
                added.add(key)

    return neighbors

# ================== 8) Build model & Load checkpoint ==================
model = SingleHeadNet(
    backbone=ResNet18Backbone(nf=64),
    num_classes=20
).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
state = ckpt["state"] if isinstance(ckpt, dict) and "state" in ckpt else ckpt
model.load_state_dict(state, strict=True)
print(f"[INFO] Loaded checkpoint from {MODEL_PATH}")

# ================== 9) Fisher computation ==================
fisher_info = compute_fisher_per_sample_ewc_weighted(model, subset_loader, DEVICE)


topk_info = get_topk_fisher_weights(fisher_info, model.state_dict(), TOP_K)
neighbors_info = extract_conv_neighbors(topk_info, model, fisher_info)

with open(SAVE_TOPK_PATH, "wb") as f:
    pickle.dump(topk_info, f)

with open(SAVE_NEIGHBORS_PATH, "wb") as f:
    pickle.dump(neighbors_info, f)

# ================== 10) Quick Stats ==================
print(f" Fisher computed from {len(subset)} samples (Tiny ImageNet, 20-class).")
print(f" Top-K total: {len(topk_info)}")

conv_topk = [e for e in topk_info if len(model.state_dict()[e['name']].shape) == 4] if topk_info else []
if len(topk_info) > 0:
    print(f" Top-K from Conv2D: {len(conv_topk)} ({100 * len(conv_topk) / len(topk_info):.2f}%)")
else:
    print(" Top-K from Conv2D: 0")

print(f" Neighbors extracted: {len(neighbors_info)}")

print("\n Fisher Statistics per-parameter:")
print(f"{'Parameter':40s} | {'Mean':>12s} | {'Min':>12s} | {'Max':>12s}")
print("-" * 85)

for name, tens in fisher_info.items():
    vals = tens.detach().cpu().view(-1)
    mean_val = vals.mean().item()
    min_val = vals.min().item()
    max_val = vals.max().item()
    print(f"{name:40s} | {mean_val:12.4e} | {min_val:12.4e} | {max_val:12.4e}")
print(" Done.")

Mounted at /content/drive
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Loaded checkpoint from /content/drive/MyDrive/ML_Project/project_files/Task1,2_tiny_imagnet/best_tinyimg_qda_20class.pth
Fisher computed from 2000 samples (Tiny ImageNet, 20-class).
 Top-K total: 900000
 Top-K from Conv2D: 891440 (99.05%)
 Neighbors extracted: 833331

 Fisher Statistics per-parameter:
Parameter                                |         Mean |          Min |          Max
-------------------------------------------------------------------------------------
backbone.conv1.weight                    |   2.7697e-05 |   6.1117e-07 |   5.2778e-04
backbone.gn1.weight                      |   1.5237e-04 |   1.8076e-06 |   2.7502e-03
backbone.gn1.bias                        |   9.2890e-05 |   4.3146e-06 |   1.0049e-03
backbone.layer1.0.conv1.weight           |   4.3545e-07 |   6.5223e-10 |   4.7010e-05
backbone.layer1.0.gn1.weight             |   5.1407e-05 |  